# Chicago Crime Analysis
## Requête 4 — Analyse spatiale & Clustering
**Auteure : Chrisa**

**Objectif :** Identifier les zones géographiques de Chicago où la criminalité se concentre,
grâce à la visualisation spatiale et aux algorithmes de clustering K-means et DBSCAN.

---

In [1]:
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from sklearn.cluster import DBSCAN, KMeans
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore")

### Chargement du dataset

On charge les données depuis l'API officielle Chicago Data Portal.
Le CSV original de M1 ne couvrait qu'une seule adresse à Bridgeport (943 lignes) —
impossible de faire une analyse spatiale représentative de toute la ville.
L'API permet de charger 10 000 crimes couvrant l'ensemble des quartiers.

Un fix SSL est appliqué automatiquement sur Mac (Python 3.12).

In [2]:
# Fonction : load_data
# Input  : limit = nombre de crimes à récupérer depuis l'API
# Output : DataFrame pandas avec colonnes renommées et Date en datetime
def load_data(limit=10000):

    url = (
        f"https://data.cityofchicago.org/resource/ijzp-q8t2.csv"
        f"?$limit={limit}"
        f"&$order=date%20DESC"
        f"&$where=latitude%20IS%20NOT%20NULL"
    )

    # Fix SSL uniquement sur Mac (Python 3.12 ne trouve pas les certificats système)
    if sys.platform == "darwin":
        import requests, io, certifi
        response = requests.get(url, verify=certifi.where())
        df = pd.read_csv(io.StringIO(response.text))
    else:
        df = pd.read_csv(url)

    # Renommage des colonnes (l'API retourne en minuscules)
    df = df.rename(columns={
        'latitude'            : 'Latitude',
        'longitude'           : 'Longitude',
        'primary_type'        : 'Primary Type',
        'arrest'              : 'Arrest',
        'date'                : 'Date',
        'location_description': 'Location Description',
        'domestic'            : 'Domestic',
        'beat'                : 'Beat',
        'ward'                : 'Ward',
        'fbi_code'            : 'FBI Code',
        'year'                : 'Year',
        'description'         : 'Description'
    })

    df["Date"] = pd.to_datetime(df["Date"], errors='coerce')
    print(f"Dataset chargé : {df.shape[0]} lignes x {df.shape[1]} colonnes")
    return df

df = load_data()
df.head()

Dataset chargé : 10000 lignes x 22 colonnes


,id,case_number,Date,block,iucr,Primary Type,Description,Location Description,Arrest,Domestic,...,Ward,community_area,FBI Code,x_coordinate,y_coordinate,Year,updated_on,Latitude,Longitude,location
0,14223030,JK286094,2026-06-08,036XX W DOUGLAS BLVD,0497,BATTERY,AGGRAVATED DOMESTIC BATTERY - OTHER DANGEROUS ...,APARTMENT,False,True,...,24,29,04B,1152476,1893218,2026,2026-06-15T15:46:40.000,41.862851,-87.715755,"\n, \n(41.862850968, -87.715755)"
1,14223973,JK287246,2026-06-08,072XX S RICHMOND ST,0486,BATTERY,DOMESTIC BATTERY SIMPLE,RESIDENCE,False,True,...,18,66,08B,1157959,1856471,2026,2026-06-15T15:46:40.000,41.761902,-87.696627,"\n, \n(41.76190249, -87.696626851)"
2,14223471,JK286728,2026-06-08,076XX S MORGAN ST,0910,MOTOR VEHICLE THEFT,AUTOMOBILE,STREET,False,False,...,17,71,07,1170965,1854265,2026,2026-06-15T15:46:40.000,41.755575,-87.649023,"\n, \n(41.755574885, -87.649022623)"
3,14224250,JK286700,2026-06-08,061XX N KIRKWOOD AVE,0930,MOTOR VEHICLE THEFT,THEFT / RECOVERY - AUTOMOBILE,STREET,False,False,...,39,12,07,1145138,1940565,2026,2026-06-15T15:46:40.000,41.992917,-87.741493,"\n, \n(41.992917396, -87.741492591)"
4,14225502,JK287707,2026-06-08,040XX W DICKENS AVE,2825,OTHER OFFENSE,HARASSMENT BY TELEPHONE,RESIDENCE,False,False,...,35,20,26,1148906,1913645,2026,2026-06-15T15:46:40.000,41.918975,-87.728331,"\n, \n(41.918974575, -87.728331482)"


### Préparation des données spatiales

On filtre les lignes sans coordonnées GPS et hors de la zone géographique de Chicago.
On convertit ensuite chaque paire (longitude, latitude) en objet `Point` géoréférencé.

**ATTENTION :** convention `Point(longitude, latitude)` — longitude en premier.
La projection `EPSG:4326` = WGS84, le standard GPS mondial.

In [3]:
# Fonction : prepare_spatial_data
# Input  : DataFrame complet
# Output : DataFrame avec uniquement les observations géolocalisables dans Chicago
def prepare_spatial_data(df):
    spatial = df.dropna(subset=["Latitude", "Longitude"]).copy()
    # Filtre géographique : Chicago est entre lat 41.6-42.1 et lon -87.9/-87.5
    return spatial[
        spatial["Latitude"].between(41.6, 42.1) &
        spatial["Longitude"].between(-87.9, -87.5)
    ].copy()


# Fonction : create_geodataframe
# Input  : DataFrame spatial avec colonnes Latitude et Longitude
# Output : GeoDataFrame avec objets Point géoréférencés (EPSG:4326)
def create_geodataframe(spatial_df):
    # Point(longitude, latitude) — longitude en premier (convention géographique)
    geometry = [Point(lon, lat) for lon, lat in
                zip(spatial_df["Longitude"], spatial_df["Latitude"])]
    gdf = gpd.GeoDataFrame(spatial_df.copy(), geometry=geometry, crs="EPSG:4326")
    print(f"GeoDataFrame créé : {len(gdf)} points | Projection : {gdf.crs}")
    return gdf


spatial_df = prepare_spatial_data(df)
gdf        = create_geodataframe(spatial_df)
gdf[["Primary Type", "Arrest", "Latitude", "Longitude", "geometry"]].head()

GeoDataFrame créé : 9968 points | Projection : EPSG:4326


,Primary Type,Arrest,Latitude,Longitude,geometry
0,BATTERY,False,41.862851,-87.715755,POINT (-87.71576 41.86285)
1,BATTERY,False,41.761902,-87.696627,POINT (-87.69663 41.7619)
2,MOTOR VEHICLE THEFT,False,41.755575,-87.649023,POINT (-87.64902 41.75557)
3,MOTOR VEHICLE THEFT,False,41.992917,-87.741493,POINT (-87.74149 41.99292)
4,OTHER OFFENSE,False,41.918975,-87.728331,POINT (-87.72833 41.91897)


## Carte de densité des crimes

Ce graphique représente la concentration géographique des crimes à Chicago.
En zoomant, on voit des zones quasi noires correspondant aux hotspots maximaux,
notamment dans le West Side et le long de la côte est.

**Calcul :** Pour chaque point de la carte, Plotly compte le nombre de crimes
dans un rayon de `radius=10` pixels et colore la zone en conséquence.

**Interprétation :** Plus la couleur est foncée, plus la densité de crimes est élevée.
La criminalité n'est pas uniformément répartie — elle forme des clusters géographiques précis.

In [4]:
# Fonction : plot_density_map
# Input  : DataFrame spatial avec colonnes Latitude et Longitude
# Output : carte de densité interactive Plotly
def plot_density_map(spatial_df):
    fig = px.density_mapbox(
        spatial_df,
        lat="Latitude",
        lon="Longitude",
        radius=10,
        center={"lat": 41.85, "lon": -87.65},
        zoom=10,
        mapbox_style="carto-positron",
        title="Densité spatiale des crimes à Chicago",
        color_continuous_scale="Reds"
    )
    fig.update_layout(height=600, margin={"r": 0, "t": 40, "l": 0, "b": 0})
    return fig

plot_density_map(spatial_df).show()

## Clustering K-means

K-means divise les crimes en `k=6` zones géographiques en regroupant les points les plus proches.
On choisit k=6 car Chicago est naturellement divisée en 6 grands secteurs
(North, Northwest, West, Central, South, Far South).

**Calcul :** L'algorithme place 6 centres, assigne chaque crime au centre le plus proche,
déplace les centres vers le milieu de leur groupe, et répète jusqu'à convergence.

**Interprétation :** Chaque couleur représente une zone géographique.
Les zones 0 et 2 concentrent le plus de crimes (plus de 2000 incidents chacune).
La zone 1 est la moins touchée (environ 1000 incidents).

**Limite :** K-means force tous les crimes dans une zone, même les crimes isolés.

In [5]:
# Fonction : apply_kmeans
# Input  : DataFrame spatial, k = nombre de clusters
# Output : DataFrame avec colonne cluster_kmeans, modèle KMeans entraîné
def apply_kmeans(spatial_df, k=6):
    coords = spatial_df[["Latitude", "Longitude"]].values
    model  = KMeans(n_clusters=k, random_state=42, n_init=10)
    df     = spatial_df.copy()
    # .astype(str) pour que Plotly affiche des couleurs distinctes (et non un dégradé)
    df["cluster_kmeans"] = model.fit_predict(coords).astype(str)
    print(f"K-means : {k} zones créées")
    print(df["cluster_kmeans"].value_counts().sort_index())
    return df, model


# Fonction : plot_kmeans_map
# Input  : DataFrame avec colonne cluster_kmeans
# Output : carte interactive colorée par zone K-means
def plot_kmeans_map(kmeans_df):
    fig = px.scatter_mapbox(
        kmeans_df,
        lat="Latitude",
        lon="Longitude",
        color="cluster_kmeans",
        center={"lat": 41.85, "lon": -87.65},
        zoom=10,
        mapbox_style="carto-positron",
        title="K-means — 6 zones de criminalité à Chicago",
        hover_data=["Primary Type", "Arrest"],
        opacity=0.6
    )
    fig.update_layout(height=600, margin={"r": 0, "t": 40, "l": 0, "b": 0})
    return fig


df_kmeans, kmeans_model = apply_kmeans(spatial_df, k=6)
plot_kmeans_map(df_kmeans).show()

K-means : 6 zones créées
cluster_kmeans
0    2323
1     954
2    2202
3    1375
4    1252
5    1862
Name: count, dtype: int64


## Clustering DBSCAN

Contrairement à K-means, DBSCAN détecte automatiquement les zones denses
sans qu'on fixe le nombre de clusters à l'avance.
Les crimes isolés sont étiquetés **-1** et apparaissent en gris sur la carte.

On a d'abord essayé OPTICS mais avec 10 000 points GPS sur Chicago,
l'algorithme mettait tout dans un seul cluster car il considérait que tout était dense.
DBSCAN est plus adapté car il permet de contrôler le rayon de recherche en kilomètres.

**Paramètres :**
- `eps=0.5/6371` : rayon de 0.5 km converti en radians (6371 = rayon de la Terre en km)
- `min_samples=50` : minimum 50 crimes dans ce rayon pour former une zone dense
- `metric="haversine"` : calcule les vraies distances GPS sur la sphère terrestre

**Interprétation :** Les hotspots colorés se concentrent dans le West Side et le long de
la côte est — confirmant exactement ce qu'on voyait sur la heatmap.
Les points gris (-1) sont des crimes géographiquement isolés, répartis sur toute la ville.

In [6]:
# Fonction : apply_dbscan
# Input  : DataFrame spatial, eps = rayon en km, min_samples = densité minimale
# Output : DataFrame avec colonne cluster_dbscan (-1 = crime isolé), modèle DBSCAN
def apply_dbscan(spatial_df, eps=0.5, min_samples=50):

    # Conversion en radians car haversine travaille en radians
    # 6371 = rayon de la Terre en km
    coords = np.radians(spatial_df[["Latitude", "Longitude"]].to_numpy())

    # eps en radians (0.5 km / 6371 km)
    # metric="haversine" calcule les vraies distances GPS sur la sphère terrestre
    model  = DBSCAN(eps=eps/6371, min_samples=min_samples,
                    metric="haversine", algorithm="ball_tree")
    result = spatial_df.copy()

    # .astype(str) pour que Plotly affiche des couleurs distinctes
    result["cluster_dbscan"] = model.fit_predict(coords).astype(str)

    nb_zones = result["cluster_dbscan"].nunique() - 1
    nb_bruit = (result["cluster_dbscan"] == "-1").sum()
    print(f"DBSCAN : {nb_zones} zones denses | {nb_bruit} crimes isolés ({nb_bruit/len(result)*100:.1f}%)")
    return result, model


# Fonction : plot_dbscan_map
# Input  : DataFrame avec colonne cluster_dbscan
# Output : carte interactive — hotspots colorés, crimes isolés en gris
def plot_dbscan_map(dbscan_df):

    # Séparer le bruit des clusters pour mieux visualiser
    bruit    = dbscan_df[dbscan_df["cluster_dbscan"] == "-1"]
    clusters = dbscan_df[dbscan_df["cluster_dbscan"] != "-1"]

    fig = go.Figure()

    # D'abord les crimes isolés en gris transparent en arrière-plan
    fig.add_trace(go.Scattermapbox(
        lat=bruit["Latitude"], lon=bruit["Longitude"],
        mode="markers",
        marker=dict(size=4, color="lightgrey", opacity=0.3),
        name="-1 (crimes isolés)"
    ))

    # Ensuite les clusters colorés par-dessus
    for cluster_id in sorted(clusters["cluster_dbscan"].unique()):
        subset = clusters[clusters["cluster_dbscan"] == cluster_id]
        fig.add_trace(go.Scattermapbox(
            lat=subset["Latitude"], lon=subset["Longitude"],
            mode="markers",
            marker=dict(size=6, opacity=0.8),
            name=f"Zone {cluster_id}"
        ))

    fig.update_layout(
        mapbox=dict(style="carto-positron",
                    center={"lat": 41.85, "lon": -87.65}, zoom=10),
        title="DBSCAN — Zones denses de criminalité à Chicago (gris = crime isolé)",
        height=600,
        margin={"r": 0, "t": 50, "l": 0, "b": 0}
    )
    return fig


df_dbscan, dbscan_model = apply_dbscan(spatial_df)
plot_dbscan_map(df_dbscan).show()

DBSCAN : 18 zones denses | 7085 crimes isolés (71.1%)


## Comparaison K-means vs DBSCAN

| Critère | K-means | DBSCAN |
|---|---|---|
| Nombre de zones | Fixé (k=6) | Automatique (18 trouvées) |
| Crimes isolés | Forcés dans une zone | Gris (-1) |
| Forme des zones | Ronde et régulière | Libre |
| Paramètre clé | k | eps (rayon en km) |

**Résultats observés :**
Les deux méthodes s'accordent sur les mêmes zones à risque — principalement le West Side
et la bande est de Chicago le long du lac Michigan.
K-means donne une vision globale en 6 grandes zones.
DBSCAN est plus précis et révèle 18 hotspots denses avec les crimes isolés en gris.

## Composition des types de crimes par zone

On enrichit le clustering K-means en regardant quels types de crimes dominent
dans chaque zone géographique.

**Interprétation :** Les clusters 0 et 2 sont les plus touchés (plus de 2000 incidents).
L'assault et la battery dominent dans toutes les zones, mais certaines ont
plus de motor vehicle theft ou de narcotics — chaque zone a un profil criminel distinct.

In [7]:
# Fonction : plot_crime_type_by_cluster
# Input  : DataFrame avec colonne cluster_kmeans
# Output : graphique barres empilées — types de crimes par zone géographique
def plot_crime_type_by_cluster(kmeans_df):
    group = (
        kmeans_df.groupby(["cluster_kmeans", "Primary Type"])
        .size()
        .reset_index(name="count")
    )
    fig = px.bar(
        group,
        x="cluster_kmeans",
        y="count",
        color="Primary Type",
        title="Composition des types de crimes par zone (K-means)",
        labels={"cluster_kmeans": "Zone", "count": "Nombre de crimes",
                "Primary Type": "Type de crime"},
        barmode="stack"
    )
    return fig

plot_crime_type_by_cluster(df_kmeans).show()

## Synthèse

- La heatmap confirme que la criminalité se concentre sur la bande est de Chicago
  et dans le West Side — elle n'est pas uniformément répartie
- K-means identifie 6 zones géographiques avec des profils criminels distincts
- DBSCAN détecte automatiquement 18 hotspots précis et révèle les crimes géographiquement isolés
- Les deux méthodes s'accordent sur les mêmes zones à risque
- Cette analyse répond à la dimension **spatiale** de notre problématique

In [8]:
# Main block — Requête 4 : Analyse spatiale
# Auteure : Chrisa
# Input   : API Chicago Data Portal (10 000 crimes)
# Output  : 4 visualisations interactives Plotly

df         = load_data()
spatial_df = prepare_spatial_data(df)
gdf        = create_geodataframe(spatial_df)

plot_density_map(spatial_df).show()

df_kmeans, kmeans_model = apply_kmeans(spatial_df, k=6)
plot_kmeans_map(df_kmeans).show()

df_dbscan, dbscan_model = apply_dbscan(spatial_df)
plot_dbscan_map(df_dbscan).show()

plot_crime_type_by_cluster(df_kmeans).show()

Dataset chargé : 10000 lignes x 22 colonnes
GeoDataFrame créé : 9968 points | Projection : EPSG:4326


K-means : 6 zones créées
cluster_kmeans
0    2323
1     954
2    2202
3    1375
4    1252
5    1862
Name: count, dtype: int64


DBSCAN : 18 zones denses | 7085 crimes isolés (71.1%)
